# Лабораторная работа №5
## Написание функций в СУБД
**База данных:** Автостоянка (Вариант 11б)  
**СУБД:** PostgreSQL 17

## Подключение

In [9]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    dbname='parking_lab',
    user='feedachyou',
    host='localhost',
    port=5432
)
conn.autocommit = True
cursor = conn.cursor()
print('Connected')

Connected


## Создание таблиц и заполнение данными

In [10]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Cars (
    License_Plate TEXT PRIMARY KEY,
    Car_Type      TEXT,
    Client_ID     INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS Parking_Access (
    License_Plate TEXT NOT NULL,
    Work_Date     DATE NOT NULL,
    Access_Permit BOOLEAN NOT NULL,
    PRIMARY KEY (License_Plate, Work_Date),
    FOREIGN KEY (License_Plate) REFERENCES Cars(License_Plate)
);

CREATE TABLE IF NOT EXISTS Payments (
    Entry_ID      SERIAL PRIMARY KEY,
    License_Plate TEXT NOT NULL,
    Work_Date     DATE NOT NULL,
    Entry_Time    TIME NOT NULL,
    Entry_Type    TEXT,
    FOREIGN KEY (License_Plate, Work_Date) REFERENCES Parking_Access(License_Plate, Work_Date)
);
""")
print('Tables created')

Tables created


In [11]:
cursor.execute("""
INSERT INTO Cars VALUES
    ('A123AA77',  'Sedan',      110),
    ('B555BB99',  'SUV',        102),
    ('C333CC199', 'SUV',        107),
    ('E777EE177', 'Sedan',      103),
    ('H888HH777', 'Motorcycle', 106),
    ('K001KK750', 'Truck',      104),
    ('M444MM97',  'SUV',        105),
    ('P999PP77',  'Truck',      108),
    ('T111TT99',  'Sedan',      109),
    ('X222XX50',  'Sedan',      101)
ON CONFLICT DO NOTHING;

INSERT INTO Parking_Access VALUES
    ('A123AA77',  '2025-08-27', TRUE),
    ('B555BB99',  '2025-06-02', TRUE),
    ('C333CC199', '2025-11-25', FALSE),
    ('E777EE177', '2025-10-15', TRUE),
    ('H888HH777', '2026-01-14', TRUE),
    ('K001KK750', '2025-12-17', FALSE),
    ('M444MM97',  '2025-11-19', TRUE),
    ('P999PP77',  '2025-10-22', TRUE),
    ('T111TT99',  '2025-09-24', TRUE),
    ('X222XX50',  '2025-10-21', TRUE)
ON CONFLICT DO NOTHING;

INSERT INTO Payments (License_Plate, Work_Date, Entry_Time, Entry_Type) VALUES
    ('A123AA77',  '2025-08-27', '08:30', 'Single'),
    ('B555BB99',  '2025-06-02', '09:50', 'Subscription'),
    ('E777EE177', '2025-10-15', '10:09', 'Single'),
    ('H888HH777', '2026-01-14', '11:22', 'Subscription'),
    ('M444MM97',  '2025-11-19', '08:25', 'Single'),
    ('P999PP77',  '2025-10-22', '08:45', 'Subscription'),
    ('T111TT99',  '2025-09-24', '13:06', 'Single'),
    ('X222XX50',  '2025-10-21', '01:07', 'Subscription'),
    ('A123AA77',  '2025-08-27', '18:04', 'Single'),
    ('B555BB99',  '2025-06-02', '20:00', 'Single')
ON CONFLICT DO NOTHING;
"""
)
print('Data inserted')

Data inserted


## Задание 1 — Скалярная функция: количество записей по типу авто

In [12]:
cursor.execute("""
CREATE OR REPLACE FUNCTION count_cars_by_type(p_type TEXT)
RETURNS INTEGER
LANGUAGE SQL AS $$
    SELECT COUNT(*)::INTEGER FROM Cars WHERE Car_Type = p_type;
$$;
""")

for car_type in ('Sedan', 'SUV', 'Truck', 'Motorcycle'):
    cursor.execute("SELECT count_cars_by_type(%s)", (car_type,))
    print(f"{car_type}: {cursor.fetchone()[0]}")

Sedan: 4
SUV: 3
Truck: 2
Motorcycle: 1


## Задание 2 — Скалярная функция: среднее количество въездов

In [13]:
cursor.execute("""
CREATE OR REPLACE FUNCTION avg_entries_per_car()
RETURNS NUMERIC
LANGUAGE SQL AS $$
    SELECT ROUND(COUNT(*)::NUMERIC / NULLIF(COUNT(DISTINCT License_Plate), 0), 2)
    FROM Payments;
$$;
""")

cursor.execute("SELECT avg_entries_per_car()")
print(f"Среднее въездов на машину: {cursor.fetchone()[0]}")

Среднее въездов на машину: 5.00


## Задание 3 — Табличная функция: список машин без допуска

In [14]:
cursor.execute("""
CREATE OR REPLACE FUNCTION cars_without_access()
RETURNS TABLE(License_Plate TEXT, Car_Type TEXT, Client_ID INTEGER)
LANGUAGE SQL AS $$
    SELECT c.License_Plate, c.Car_Type, c.Client_ID
    FROM Cars c
    JOIN Parking_Access pa ON c.License_Plate = pa.License_Plate
    WHERE pa.Access_Permit = FALSE;
$$;
""")

df = pd.read_sql("SELECT * FROM cars_without_access()", conn)
df

/var/folders/zb/dsr0j93j6sgd8wplkp6h6x3m0000gn/T/ipykernel_9717/1756559128.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM cars_without_access()", conn)


,license_plate,car_type,client_id
0,C333CC199,SUV,107
1,K001KK750,Truck,104


## Задание 4 — Функция с проверкой: добавление записи

In [15]:
cursor.execute("""
CREATE OR REPLACE FUNCTION add_car_if_not_exists(
    p_plate TEXT, p_type TEXT, p_client INTEGER
) RETURNS BOOLEAN
LANGUAGE plpgsql AS $$
BEGIN
    IF EXISTS (SELECT 1 FROM Cars WHERE License_Plate = p_plate) THEN
        RETURN FALSE;
    END IF;
    INSERT INTO Cars VALUES (p_plate, p_type, p_client);
    RETURN TRUE;
END;
$$;
""")

cursor.execute("SELECT add_car_if_not_exists(%s, %s, %s)", ('Z999ZZ99', 'Sedan', 200))
print(f"Добавлена новая: {cursor.fetchone()[0]}")

cursor.execute("SELECT add_car_if_not_exists(%s, %s, %s)", ('A123AA77', 'Sedan', 110))
print(f"Уже существует:  {cursor.fetchone()[0]}")

Добавлена новая: True
Уже существует:  False


## Задание 5 — Функция с проверкой: удаление записи

In [16]:
cursor.execute("""
CREATE OR REPLACE FUNCTION delete_car_if_exists(p_plate TEXT)
RETURNS BOOLEAN
LANGUAGE plpgsql AS $$
BEGIN
    IF NOT EXISTS (SELECT 1 FROM Cars WHERE License_Plate = p_plate) THEN
        RETURN FALSE;
    END IF;
    DELETE FROM Cars WHERE License_Plate = p_plate;
    RETURN TRUE;
END;
$$;
""")

cursor.execute("SELECT delete_car_if_exists(%s)", ('Z999ZZ99',))
print(f"Удалена Z999ZZ99:  {cursor.fetchone()[0]}")

cursor.execute("SELECT delete_car_if_exists(%s)", ('Z999ZZ99',))
print(f"Повторное удаление: {cursor.fetchone()[0]}")

Удалена Z999ZZ99:  True
Повторное удаление: False
